# OCVWorkChain: DFT (GGA) submission

Computes the average, low-SOC and high-SOC open-circuit voltages of a cathode at the plain GGA (PBEsol) level.

Requires aiida-open_circuit_voltage >= 0.7 with aiida-quantumespresso 5.x (PwRelax namespaces: base_relax; no base_final_scf / volume_convergence).

The example structures used below are bundled with the repository as an AiiDA archive.
Import them once with

```
verdi archive import test/structures.aiida
```

or load your own structure instead, e.g. `orm.StructureData(ase=ase.io.read('my_cathode.cif'))`.

## Loading libraries

In [ ]:
from aiida import load_profile, orm
## Indicate your profile name here
your_profile_name = 'develop'
load_profile(your_profile_name)
from aiida.plugins import WorkflowFactory
from aiida.engine import submit

## Loading the code and structure

In [ ]:
## AiiDA code that will run Quantum ESPRESSO on the cluster
code = orm.load_code(label='pw@your_computer')

## default resources — adjust to your cluster
time, num_machines, num_mpiprocs_per_machine, num_cores_per_mpiproc, npool = 83200, 2, 128, 1, 8

## Load one of the 3 bundled test structures here (import test/structures.aiida first)
structure = orm.load_node('096d9d96-7f66-442b-a27f-2660572808ea') # LiCoO2
# structure = orm.load_node('3dd4a60a-a5d5-48b6-a8b1-3e082664622a') # LiFePO4
# structure = orm.load_node('c0537852-7300-43ea-af99-7813eea8a167') # MgMo3S4

## Load either the Li or Mg bulk cation structure here depending on the above ``structure``
bulk_cation_structure = orm.load_node('faea3c48-789f-4076-af73-0cf9242bb7b2') # Li
# bulk_cation_structure = orm.load_node('df0a8dda-3d84-44d6-8b97-069ee8d0a62e') # Mg

## Launching the OCVWorkChain

In [ ]:
## Defining the OCVWorkChain with some suggested parameters

## overrides defining pseudopotentials
## aiida-qe 5: PwRelax has a single 'base_relax' namespace (no 'base_final_scf')
## the SCF-only runs on pre-relaxed unitcells are derived from base_relax by the OCV workchain.
overrides = {"ocv_relax":{
                "base_relax":{
                    "pseudo_family": "SSSP/1.3/PBEsol/efficiency",
                    "pw":{
                        "parallelization":{
                            "npool": npool},}}},
            "scf":{
                "pseudo_family": "SSSP/1.3/PBEsol/efficiency",
                "pw":{
                    "parallelization":{
                        "npool": int(npool/num_machines)},}},}

def submit_OCV_workchain(structure, code, bulk_cation_structure=None, discharged_unitcell_relaxed=None,
                         charged_unitcell_relaxed=None, protocol='fast', time=time,
                         num_machines=num_machines, num_mpiprocs_per_machine=num_mpiprocs_per_machine,
                         num_cores_per_mpiproc=num_cores_per_mpiproc):
    """Return a configured plain-GGA OCVWorkChain builder.

    protocol: 'fast' (testing), 'balanced' (default quality) or 'stringent' (high precision).
    """
    OCVWorkChain = WorkflowFactory('quantumespresso.ocv.ocvwc')

    builder = OCVWorkChain.get_builder_from_protocol(
        code=code, structure=structure, bulk_cation_structure=bulk_cation_structure,
        discharged_unitcell_relaxed=discharged_unitcell_relaxed,
        charged_unitcell_relaxed=charged_unitcell_relaxed,
        overrides=overrides, protocol=protocol)

    builder.update({'clean_workdir': orm.Bool(False)})

    # the endpoint SCF inherits everything (resources, electron_maxstep) from base_relax
    pw_dict = builder.ocv_relax['base_relax']['pw']
    pw_dict['metadata']['options']['max_wallclock_seconds'] = time
    pw_dict['metadata']['options']['resources']['num_machines'] = num_machines
    pw_dict['metadata']['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
    pw_dict['metadata']['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
    pw_dict['parameters']['ELECTRONS']['electron_maxstep'] = 100

    # Keep a small value as it finishes in a few minutes even on a small cluster
    if bulk_cation_structure:
        builder.scf['pw']['metadata']['options']['max_wallclock_seconds'] = 1800
        builder.scf['pw']['metadata']['options']['resources']['num_machines'] = 1
        builder.scf['pw']['metadata']['options']['resources']['num_mpiprocs_per_machine'] = num_mpiprocs_per_machine
        builder.scf['pw']['metadata']['options']['resources']['num_cores_per_mpiproc'] = num_cores_per_mpiproc
        builder.scf['pw']['parameters']['ELECTRONS']['electron_maxstep'] = 100
        builder.scf['pw']['parallelization'] = orm.Dict(dict={'npool': int(npool/num_machines)})

    builder.ocv_parameters['distance'] = 8.0

    # Change to 'Mg' in case you use MgMo3S4 as the structure; 'Li' is the default.
    # If omitted, the cation is inferred when the structure contains exactly one supported cation.
    builder.ocv_parameters['cation'] = 'Li'

    return builder

In [ ]:
## Submitting the OCVWorkChain
builder = submit_OCV_workchain(structure=structure, code=code,
                               bulk_cation_structure=bulk_cation_structure)
node = submit(builder)
print(f'Submitted OCVWorkChain PK={node.pk}')

## Results

In [ ]:
## Once the workchain has finished, inspect the voltages
node = orm.load_node(node.uuid)   # the OCVWorkChain printed at submission
print(node.outputs.open_circuit_voltages.get_dict())